# Monte Carlo Simulation for Project Cost Risk

**Author:** Alejandro Vazquez

When you estimate a project budget by adding up your "best guess" for each cost, you get a single number — but reality is uncertain, and that single number hides the risk. **Monte Carlo simulation** solves this: instead of one guess per item, we model each cost as a *range* of possibilities, then simulate the project thousands of times to see the full distribution of outcomes.

This notebook estimates the total cost of a project, quantifies the **probability of going over budget**, and identifies **which cost item drives the most risk**.

## 1. Setup

We only need three standard libraries — all pre-installed in Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # fixed seed = reproducible results
N = 50_000                        # number of simulated projects

## 2. Define the cost items

For each item we use a **three-point estimate**: the minimum, the most likely, and the maximum cost. This is a standard way to capture uncertainty when you don't have historical data — you just ask "what's the best case, the expected case, and the worst case?"

We model each item with a **triangular distribution**, which peaks at the "most likely" value and tapers toward the min and max.

In [ ]:
# item: (minimum, most likely, maximum)
items = {
    "Labor":          (20000, 28000, 45000),
    "Software/Tools": ( 8000, 10000, 15000),
    "Equipment":      ( 5000,  7000, 12000),
    "Subcontractors": (10000, 15000, 28000),
    "Contingency":    ( 3000,  5000,  9000),
}

BUDGET = 70000   # the budget we want to test against

## 3. Run the simulation

For every item we draw `N` random samples from its triangular distribution. Each simulated project sums one sample from each item, giving us `N` possible total costs.

The whole thing is **vectorized** with NumPy, so 50,000 simulations run instantly.

In [ ]:
samples = {name: rng.triangular(lo, mode, hi, N)
           for name, (lo, mode, hi) in items.items()}

df = pd.DataFrame(samples)      # each column = one cost item, each row = one simulated project
total = df.sum(axis=1).values   # total cost per simulated project

df.head()

## 4. Read the results

Instead of a single number, we now have a distribution. The most useful summaries are **percentiles**:

- **P10** — 10% of outcomes fall below this (an optimistic case)
- **P50** — the median (half above, half below)
- **P90** — 90% fall below this (a conservative/safe estimate)

And the key business question: **what's the probability of exceeding the budget?**

In [ ]:
p10, p50, p90 = np.percentile(total, [10, 50, 90])
prob_over = (total > BUDGET).mean() * 100

print(f"Mean total cost : {total.mean():>10,.0f}")
print(f"P10  (optimistic): {p10:>10,.0f}")
print(f"P50  (median)    : {p50:>10,.0f}")
print(f"P90  (safe)      : {p90:>10,.0f}")
print(f"\nProbability of exceeding budget ({BUDGET:,}): {prob_over:.1f}%")

## 5. Visualize the distribution

A histogram of the 50,000 outcomes tells the whole story at a glance: where costs cluster, and how much of the distribution sits past the budget line.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(total, bins=60, color="#4C78A8", alpha=0.85, edgecolor="white", linewidth=0.4)

for val, lab, col in [(p10, "P10", "#59A14F"),
                      (p50, "P50 (median)", "#000000"),
                      (p90, "P90", "#E15759")]:
    ax.axvline(val, color=col, linestyle="--", linewidth=1.8, label=f"{lab}: {val:,.0f}")
ax.axvline(BUDGET, color="#F28E2B", linewidth=2.4, label=f"Budget: {BUDGET:,.0f}")

ax.set_title("Project Cost Risk - Monte Carlo Simulation (50,000 runs)", fontweight="bold", fontsize=13)
ax.set_xlabel("Total project cost"); ax.set_ylabel("Frequency")
ax.legend(frameon=False)
ax.text(0.98, 0.5, f"P(exceeding budget) = {prob_over:.0f}%", transform=ax.transAxes,
        ha="right", fontsize=12, fontweight="bold", color="#E15759")
plt.tight_layout()
plt.savefig("cost_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Sensitivity: which item drives the risk?

Not all costs contribute equally to the uncertainty. By measuring how strongly each item **correlates** with the total, we find the biggest risk drivers — the items worth negotiating or padding first. This is called a **tornado chart**.

In [ ]:
corrs = {name: np.corrcoef(df[name], total)[0, 1] for name in items}
corrs = dict(sorted(corrs.items(), key=lambda x: abs(x[1])))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(list(corrs.keys()), list(corrs.values()), color="#4C78A8", alpha=0.9)
ax.set_title("Sensitivity - Which item drives the risk?", fontweight="bold", fontsize=13)
ax.set_xlabel("Correlation with total cost")
for i, (k, v) in enumerate(corrs.items()):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig("tornado.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Interpretation

Even though the **most likely** cost of each item sums to about 65,000, the simulation shows the **mean outcome is higher** and there is a substantial probability of exceeding the 70,000 budget. This happens because costs can overrun more than they can underrun — a pattern a single-point estimate completely misses.

The tornado chart points to the item with the highest correlation as the main risk driver: the first place to focus on tighter estimates, negotiation, or contingency.

**This same technique applies to any uncertain quantity** — project schedules, financial forecasts, investment returns, or scientific/experimental planning. Swap the items and distributions, and the workflow stays the same.